<a href="https://colab.research.google.com/github/Cloud-Architect-Emma/minillm-research-lab/blob/main/02_transformer_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from urllib.request import urlretrieve

torch.manual_seed(42)

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

data_path = data_dir / "tiny_shakespeare.txt"

url = (
    "https://raw.githubusercontent.com/"
    "karpathy/char-rnn/master/data/"
    "tinyshakespeare/input.txt"
)

if not data_path.exists():
    urlretrieve(url, data_path)

text = data_path.read_text(encoding="utf-8")

chars = sorted(set(text))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[int(i)] for i in ids)

all_tokens = torch.tensor(
    encode(text),
    dtype=torch.long
)

split = int(0.9 * len(all_tokens))
train_data = all_tokens[:split]
val_data = all_tokens[split:]

print("Vocabulary size:", vocab_size)
print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))
print("Sample:", repr(decode(train_data[:30])))

assert decode(encode("Hello")) == "Hello"
print("Dataset ready!")


Vocabulary size: 65
Training tokens: 1003854
Validation tokens: 111540
Sample: 'First Citizen:\nBefore we proce'
Dataset ready!


In [2]:

block_size = 32
n_embd = 64
n_head = 4
n_layer = 2
dropout = 0.1

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

assert n_embd % n_head == 0

print("Device:", device)
print("Context window:", block_size)
print("Embedding size:", n_embd)
print("Attention heads:", n_head)
print("Transformer blocks:", n_layer)


Device: cpu
Context window: 32
Embedding size: 64
Attention heads: 4
Transformer blocks: 2


In [3]:

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(
            n_embd, head_size, bias=False
        )
        self.query = nn.Linear(
            n_embd, head_size, bias=False
        )
        self.value = nn.Linear(
            n_embd, head_size, bias=False
        )

        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(block_size, block_size)
            )
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        scores = (
            q @ k.transpose(-2, -1)
        ) * (k.shape[-1] ** -0.5)

        scores = scores.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        weights = F.softmax(
            scores, dim=-1
        )
        weights = self.dropout(weights)

        output = weights @ v

        return output

print("Single attention head defined!")


Single attention head defined!


In [4]:

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()

        self.heads = nn.ModuleList([
            Head(head_size)
            for _ in range(num_heads)
        ])

        self.proj = nn.Linear(
            num_heads * head_size,
            n_embd
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        combined = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )

        return self.dropout(
            self.proj(combined)
        )

print("Multi-head attention defined!")


Multi-head attention defined!


In [5]:

class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

print("Feed-forward network defined!")


Feed-forward network defined!


In [6]:

class Block(nn.Module):
    def __init__(self):
        super().__init__()

        head_size = n_embd // n_head

        self.sa = MultiHeadAttention(
            n_head, head_size
        )

        self.ffwd = FeedForward()

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))

        return x

print("Transformer block defined!")


Transformer block defined!


In [7]:

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(
            vocab_size, n_embd
        )

        self.position_embedding_table = nn.Embedding(
            block_size, n_embd
        )

        self.blocks = nn.Sequential(
            *[Block() for _ in range(n_layer)]
        )

        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(
            n_embd, vocab_size
        )

    def forward(self, idx, targets=None):
        B, T = idx.shape

        if T > block_size:
            raise ValueError(
                "Input exceeds context window"
            )

        token_embeddings = (
            self.token_embedding_table(idx)
        )

        position_embeddings = (
            self.position_embedding_table(
                torch.arange(
                    T, device=idx.device
                )
            )
        )

        x = token_embeddings + position_embeddings

        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            B, T, C = logits.shape

            loss = F.cross_entropy(
                logits.reshape(B * T, C),
                targets.reshape(B * T)
            )

        return logits, loss

model = MiniGPT().to(device)

parameter_count = sum(
    p.numel() for p in model.parameters()
)

print("MiniGPT created!")
print("Trainable parameters:", parameter_count)
print("Device:", device)


MiniGPT created!
Trainable parameters: 110145
Device: cpu


In [8]:

batch_size = 4

x = torch.randint(
    0, vocab_size,
    (batch_size, block_size),
    device=device
)

y = torch.randint(
    0, vocab_size,
    (batch_size, block_size),
    device=device
)

model.eval()

with torch.no_grad():
    logits, loss = model(x, y)

print("Input shape:", x.shape)
print("Output shape:", logits.shape)
print("Loss:", loss.item())

assert logits.shape == (
    batch_size, block_size, vocab_size
)

assert torch.isfinite(loss)

print("PASS: Forward pass works!")


Input shape: torch.Size([4, 32])
Output shape: torch.Size([4, 32, 65])
Loss: 4.288885593414307
PASS: Forward pass works!


In [9]:

model.train()
model.zero_grad(set_to_none=True)

logits, loss = model(x, y)

loss.backward()

parameters_with_gradients = sum(
    p.grad is not None
    for p in model.parameters()
)

total_parameters = sum(
    1 for p in model.parameters()
)

print(
    "Parameter tensors with gradients:",
    parameters_with_gradients
)

print(
    "Total parameter tensors:",
    total_parameters
)

assert parameters_with_gradients == total_parameters

print("PASS: Gradients calculated successfully!")


Parameter tensors with gradients: 50
Total parameter tensors: 50
PASS: Gradients calculated successfully!


In [10]:

model.eval()

original = torch.tensor(
    [encode("The cat")],
    device=device
)

modified = original.clone()

modified[0, -1] = stoi["s"]

with torch.no_grad():
    original_logits, _ = model(original)
    modified_logits, _ = model(modified)

same_earlier_outputs = torch.allclose(
    original_logits[:, :-1, :],
    modified_logits[:, :-1, :],
    atol=1e-6
)

print(
    "Earlier outputs unchanged:",
    same_earlier_outputs
)

assert same_earlier_outputs

print("PASS: Causal masking works!")


Earlier outputs unchanged: True
PASS: Causal masking works!


In [11]:

context = train_data[:block_size].unsqueeze(0)
targets = train_data[1:block_size+1].unsqueeze(0)

context = context.to(device)
targets = targets.to(device)

model.eval()

with torch.no_grad():
    logits, loss = model(context, targets)

print("Input text:")
print(repr(decode(context[0].tolist())))

print("\nTarget text:")
print(repr(decode(targets[0].tolist())))

print("\nInitial loss:", round(loss.item(), 4))

assert logits.shape == (1, block_size, vocab_size)
assert torch.isfinite(loss)

print("\nPASS: Real-data forward pass works!")


Input text:
'First Citizen:\nBefore we proceed'

Target text:
'irst Citizen:\nBefore we proceed '

Initial loss: 4.2596

PASS: Real-data forward pass works!


In [12]:

@torch.no_grad()
def generate(model, idx, max_new_tokens):
    model.eval()

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]

        logits, _ = model(idx_cond)

        last_token_logits = logits[:, -1, :]

        probabilities = F.softmax(
            last_token_logits, dim=-1
        )

        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        idx = torch.cat(
            (idx, next_token),
            dim=1
        )

    return idx


start = torch.tensor(
    [encode("The ")],
    dtype=torch.long,
    device=device
)

generated = generate(
    model,
    start,
    max_new_tokens=100
)

print(decode(generated[0].tolist()))

assert generated.shape[1] == 104

print("\nPASS: Text generation works!")


The LoGVECxy-Kx.LI&hHGgwY-3CF
.htTYoIY3ruRORbyXn3UAiXLf:

pC&LcR?Q.Wufdu;ibXLg!lLI?:iQQGXaTJkIdwLSRxbdkL

PASS: Text generation works!


In [13]:

import json
from pathlib import Path

output_dir = Path("artifacts")
output_dir.mkdir(exist_ok=True)

config = {
    "vocab_size": vocab_size,
    "block_size": block_size,
    "n_embd": n_embd,
    "n_head": n_head,
    "n_layer": n_layer,
    "dropout": dropout,
    "seed": 42
}

with open(
    output_dir / "model_config.json",
    "w"
) as file:
    json.dump(config, file, indent=2)

with open(
    output_dir / "vocab.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(chars, file, ensure_ascii=False)

torch.save(
    model.state_dict(),
    output_dir / "minigpt_untrained.pt"
)

print("Initial model weights saved!")
print("Configuration saved!")
print("Vocabulary saved!")


Initial model weights saved!
Configuration saved!
Vocabulary saved!


In [15]:

import shutil
from google.colab import files

shutil.make_archive(
    "minillm_lab2_artifacts",
    "zip",
    root_dir=".",
    base_dir="artifacts"
)

files.download("minillm_lab2_artifacts.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>